In [ ]:
        }
    }

tool_schema = to_schema(get_current_weather)
pprint(tool_schema)



# ---------------------------------------------------------
# Provide the tool schema to the model
# ---------------------------------------------------------
# Goal:
#   Give the model a "menu" of available tools so it can choose
#   which one to call based on the user’s question.
#
# Steps:
#   1. Add an extra system message (e.g., name="tool_spec")
#      containing the JSON schema(s) of your tools.
#   2. Include SYSTEM_PROMPT and the user question as before.
#   3. Send the messages to the model (e.g., gemma3:1b).
#   4. Print the raw model output to see if it picks the right tool.
#
# Expected:
#   The model should produce a structured TOOL_CALL indicating
#   which tool to use and with what arguments.
# ---------------------------------------------------------

messages=[
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "system", "name": "tool_spec", "content": json.dumps([tool_schema])},
    {"role": "user", "content": "Is Boston warmer today or Seattle?"}
]
MODEL = "gemma3:1b" # doesn't produce two calls
# MODEL = "llama3.2:3b" # produces two calls
raw2 = client.chat.completions.create(
    model=MODEL, 
    messages=messages,
    temperature=0.5,
)
print(raw2.choices[0].message.content)

In [ ]:
# ---------------------------------------------------------
# Step 1: Implement the tool
# ---------------------------------------------------------
# Your goal: give the model a way to access weather information.
# You can either:
#   (a) Call a real weather API (for example, OpenWeatherMap), or
#   (b) Create a dummy function that returns a fixed response (e.g., "It is 23°C and sunny in San Francisco.")
#
# Requirements:
#   • The function should be named `get_current_weather`
#   • It should take two arguments:
#         - city: str
#         - unit: str = "celsius"
#   • Return a short, human-readable sentence describing the weather.
#
# Example expected behavior:
#   get_current_weather("San Francisco") → "It is 23°C and sunny in San Francisco."
#
def get_current_weather(city: str, unit: str = "celsius") -> str:
    """Return the current weather as a human-readable sentence."""
    return f"It is 23°{unit[0].upper()} and sunny in {city}." # no real API call to stay offline-friendly


# ---------------------------------------------------------
# Step 2: Create the prompt for the LLM to call tools
# ---------------------------------------------------------
# Goal:
#   Build the system and user prompts that instruct the model when and how
#   to use your tool (`get_current_weather`).
#
# What to include:
#   • A SYSTEM_PROMPT that tells the model about the tool use and describe the tool
#   • A USER_QUESTION with a user query that should trigger the tool.
#       Example: "What is the weather in San Diego today?"

# Try experimenting with different system and user prompts
# ---------------------------------------------------------

SYSTEM_PROMPT = (
    "You are an assistant that can call tools. "
    "When the user asks something requiring fresh data, respond **only** with a JSON like: "
    'TOOL_CALL:{"name": <tool_name>, "args": { ... }}.'
)

TOOLS_SPEC = """
You can call exactly one tool:
- name: get_current_weather
  description: Return the current weather for a city.
  arguments:
    city: string
    unit: "celsius" | "fahrenheit"  (optional, default "celsius")
"""

USER_QUESTION = "What is the weather in San Diego today?"
# USER_QUESTION = "What is your name?"




# ---------------------------------------------------------
# Step 4: Parse the LLM output and call the tool
# ---------------------------------------------------------
# Task:
#   Detect when the model requests a tool, extract its name and arguments,
#   and execute the corresponding function.
#
# Steps:
#   1. Search for the text pattern "TOOL_CALL:{...}" in the model output.
#   2. Parse the JSON inside it to get the tool name and args.
#   3. Call the matching function (e.g., get_current_weather).
#
# Expected:
#   You should see a line like:
#       Calling tool `get_current_weather` with args {'city': 'San Diego'}
#       Result: It is 23°C and sunny in San Diego.
# ---------------------------------------------------------

import re, json

pattern = r'TOOL_CALL:\s*({.*})'
match = re.search(pattern, raw, flags=re.DOTALL)
if match:
    call_json = json.loads(match.group(1))
    fn_name   = call_json["name"]
    args      = call_json.get("args", {})
    print(f"\n Calling tool `{fn_name}` with args {args}")
    result = globals()[fn_name](**args)
    print("Result:", result)
else:
    print("No tool call detected")



# ---------------------------------------------------------
# Generate a JSON schema for a tool automatically
# ---------------------------------------------------------
#
# Steps:
#   1. Use `inspect.signature` to get function parameters.
#   2. For each argument, record its name, type, and description.
#   3. Build a schema containing:
#   4. Test your helper on `get_current_weather` and print the result.
#
# Expected:
#   A dictionary describing the tool (its name, args, and types).
# ---------------------------------------------------------

from pprint import pprint
import inspect


def to_schema(fn):
    sig = inspect.signature(fn)
    props = {
        name: {
            'type': 'string' if param.annotation is str else 'number',
            'description': f"Argument {name}"
        }
        for name, param in sig.parameters.items()
    }
    return {
        'name': fn.__name__,
        'description': (fn.__doc__ or '').strip().split('\n')[0],
        'parameters': {
            'type': 'object',
            'properties': props,
            'required': [n for n, p in sig.parameters.items() if p.default is inspect._empty]
        }
    }

tool_schema = to_schema(get_current_weather)
pprint(tool_schema)



# ---------------------------------------------------------
# Provide the tool schema to the model
# ---------------------------------------------------------
# Goal:
#   Give the model a "menu" of available tools so it can choose
#   which one to call based on the user’s question.
#
# Steps:
#   1. Add an extra system message (e.g., name="tool_spec")
#      containing the JSON schema(s) of your tools.
#   2. Include SYSTEM_PROMPT and the user question as before.
#   3. Send the messages to the model (e.g., gemma3:1b).
#   4. Print the raw model output to see if it picks the right tool.
#
# Expected:
#   The model should produce a structured TOOL_CALL indicating
#   which tool to use and with what arguments.
# ---------------------------------------------------------

messages=[
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "system", "name": "tool_spec", "content": json.dumps([tool_schema])},
    {"role": "user", "content": "Is Boston warmer today or Seattle?"}
]
MODEL = "gemma3:1b" # doesn't produce two calls
# MODEL = "llama3.2:3b" # produces two calls
raw2 = client.chat.completions.create(
    model=MODEL, 
    messages=messages,
    temperature=0.5,
)
print(raw2.choices[0].message.content)

In [ ]:
# ---------------------------------------------------------
# Step 1: Define tools for LangChain
# ---------------------------------------------------------

from langchain.tools import tool

def get_current_weather(city: str, unit: str = "celsius") -> str:
    """Return the current weather as a human-readable sentence."""
    return f"It is 23°{unit[0].upper()} and sunny in {city}." # no real API call to stay offline-friendly

@tool
def get_weather(city: str) -> str:
    """Get current weather and return results"""
    return get_current_weather(city)

# ---------------------------------------------------------
# Step 2: Initialize the LangChain Agent
# ---------------------------------------------------------

from langchain_community.chat_models import ChatOllama
from langchain.agents import initialize_agent, Tool, AgentType

MODEL = "gemma3:1b"
# MODEL = "llama3.2:3b"
# MODEL = "gemma3"
# MODEL = "llama2:7b"
llm = ChatOllama(model=MODEL, temperature=0)
tools = [get_weather]

agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
)

agent.invoke({"input": "Do I need an umbrella in Seattle today?"})